In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
with open("../data/processed/gfg_chunks.txt", "r") as f:
    chunks = [line.strip() for line in f]
faiss_index = faiss.read_index("../data/embeddings/vector_index.faiss")


In [ ]:
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import re

# Configuration
PATHS = {
    "raw_chunks": "../data/raw/gfg_content.txt",  # Original source file
    "processed_chunks": "../data/processed/gfg_chunks.txt",
    "faiss_index": "../data/embeddings/vector_index.faiss"
}

# Initialize models
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
generation_model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def preprocess_text(text):
    """Clean and normalize text"""
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    text = re.sub(r'[^\w\s.,;?!-]', '', text)  # Remove special chars
    return text.strip()

def create_chunks_and_index():
    """Recreate both chunks and index to ensure synchronization"""
    try:
        # Load raw content
        with open(PATHS["raw_chunks"], "r", encoding='utf-8') as f:
            content = f.read()
        
        # Split into meaningful chunks (adjust as needed)
        chunks = [preprocess_text(chunk) for chunk in content.split('\n\n') if chunk.strip()]
        chunks = [chunk for chunk in chunks if 50 < len(chunk.split()) < 500]  # Length filter
        
        # Save processed chunks
        os.makedirs(os.path.dirname(PATHS["processed_chunks"]), exist_ok=True)
        with open(PATHS["processed_chunks"], "w", encoding='utf-8') as f:
            f.write("\n".join(chunks))
        
        # Create embeddings and index
        embeddings = embedding_model.encode(chunks)
        
        os.makedirs(os.path.dirname(PATHS["faiss_index"]), exist_ok=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)
        faiss.write_index(index, PATHS["faiss_index"])
        
        print(f"Created {len(chunks)} chunks and corresponding index")
        return True
    
    except Exception as e:
        print(f"Error recreating resources: {str(e)}")
        return False

def load_resources():
    """Load chunks and FAISS index with validation"""
    try:
        # Verify both files exist
        if not all(os.path.exists(path) for path in [PATHS["processed_chunks"], PATHS["faiss_index"]]):
            print("Missing resource files, recreating...")
            if not create_chunks_and_index():
                return None, None
        
        # Load chunks
        with open(PATHS["processed_chunks"], "r", encoding='utf-8') as f:
            chunks = [line.strip() for line in f if line.strip()]
        
        # Load index
        index = faiss.read_index(PATHS["faiss_index"])
        
        # Quick validation
        if index.ntotal != len(chunks):
            print("Index/chunk mismatch detected, recreating...")
            if create_chunks_and_index():
                return load_resources()  # Retry after recreation
            return None, None
        
        return chunks, index
    
    except Exception as e:
        print(f"Error loading resources: {str(e)}")
        return None, None

# [Keep all other functions from previous implementation: retrieve_relevant_chunks, generate_answer, rag_chatbot]

if __name__ == "__main__":
    # First ensure resources exist and are synchronized
    print("Initializing resources...")
    chunks, faiss_index = load_resources()
    
    if chunks and faiss_index:
        test_questions = [
            "What is machine learning?",
            "Explain neural networks",
            "How does gradient descent work?"
        ]
        
        for question in test_questions:
            print(f"\n{'='*50}")
            print(f"Question: {question}")
            answer, chunks = rag_chatbot(question)
            print(f"\nAnswer: {answer}")
            
            if chunks:
                print("\nRelevant chunks used:")
                for i, chunk in enumerate(chunks, 1):
                    print(f"\nChunk {i} ({len(chunk.split())} words): {chunk[:150]}...")
    else:
        print("Failed to initialize resources. Please check your data files.")

In [ ]:
test_questions = [
    "What is machine learning?",
    "Explain neural networks",
    "How does gradient descent work?"
]

for question in test_questions:
    print(f"\n{'='*50}")
    print(f"Question: {question}")
    answer, chunks = rag_chatbot(question)
    print(f"\nAnswer: {answer}")
    
    if chunks:
        print("\nRelevant chunks used:")
        for i, chunk in enumerate(chunks, 1):
            print(f"\nChunk {i} ({len(chunk.split())} words): {chunk[:150]}...")


Question: What is machine learning?
Error loading resources: Index size (517) doesn't match chunks count (465)

Answer: System error: Failed to load required data.

Question: Explain neural networks
Error loading resources: Index size (517) doesn't match chunks count (465)

Answer: System error: Failed to load required data.

Question: How does gradient descent work?
Error loading resources: Index size (517) doesn't match chunks count (465)

Answer: System error: Failed to load required data.


In [ ]:
import os
import faiss
from sentence_transformers import SentenceTransformer
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Configuration
PATHS = {
    "chunks": "../data/processed/gfg_chunks.txt",
    "faiss_index": "../data/embeddings/vector_index.faiss"
}

# Initialize models
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
generation_model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def load_resources():
    """Load chunks and FAISS index with error handling"""
    try:
        # Load text chunks
        with open(PATHS["chunks"], "r", encoding='utf-8') as f:
            chunks = [line.strip() for line in f if line.strip()]
        
        # Load FAISS index
        if os.path.exists(PATHS["faiss_index"]):
            faiss_index = faiss.read_index(PATHS["faiss_index"])
            return chunks, faiss_index
        else:
            raise FileNotFoundError(f"FAISS index not found at {PATHS['faiss_index']}")
    
    except Exception as e:
        print(f"Error loading resources: {e}")
        return None, None

def retrieve_relevant_chunks(question, index, chunks, k=3):
    """Retrieve top-k relevant chunks"""
    question_embedding = embedding_model.encode([question])
    distances, indices = index.search(question_embedding, k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

def generate_answer(question, relevant_chunks):
    """Generate answer using GPT-2"""
    context = "\n\n".join(relevant_chunks)[:3000]  # Truncate long context
    
    prompt = f"""Context: {context}\n\nQuestion: {question}\n\nAnswer:"""
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    
    with torch.no_grad():
        outputs = generation_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=1
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()

def rag_chatbot(question):
    """Main RAG chatbot function"""
    chunks, faiss_index = load_resources()
    if not chunks or not faiss_index:
        return "System initialization failed. Missing data files.", []
    
    relevant_chunks = retrieve_relevant_chunks(question, faiss_index, chunks)
    relevant_chunks = [chunk for chunk in relevant_chunks if chunk.strip()]
    
    if not relevant_chunks:
        return "I couldn't find relevant information to answer your question.", []
    
    answer = generate_answer(question, relevant_chunks)
    return answer, relevant_chunks


In [ ]:
question = "What is machine learning?"
answer, chunks = rag_chatbot(question)

print(f"Question: {question}")
print(f"Answer: {answer}")

if chunks:
    print("\nRelevant chunks used:")
    for i, chunk in enumerate(chunks, 1):
        print(f"\nChunk {i}: {chunk[:200]}...")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is machine learning?
Answer: Machine learning is a way of thinking about machine learning and it is very important for a researcher. Machine learning is a way of thinking about machine learning and it is very important for a researcher. Machine learning is a way of thinking about machine learning and it is very important for a researcher. It is the key to the ability to identify and predict the right conditions and processes in a network. It is also the key to the ability to identify and predict the right processes in a network. It is also the key to the ability to identify and predict the right processes in a network. It is also the key to the ability to identify and predict the right processes in a network. It is also the key to the ability to identify and predict the right processes in

Relevant chunks used:

Chunk 1: Similar Topics Experiences 17k+ articles GATE 3.3k+ articles GBlog 3.1k+ articles Computer Subject 1.7k+ articles GATE CS 1.6k+ articles Competitive Exa

In [ ]:
import numpy as np
knowledge_base = np.load("D:\RAG-chatbot\data\embeddings\gfg_embeddings_mpnet_2.npz")['chunks']

In [ ]:
len(knowledge_base[0].split())

400

In [18]:
import faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-mpnet-base-v2')

In [21]:
knowledge_base = np.array([
    "ML_DEFINITION: Machine learning is a branch of artificial intelligence that develops systems capable of learning from data, identifying patterns, and making decisions with minimal human intervention. Key approaches include supervised learning (using labeled data), unsupervised learning (finding hidden patterns), and reinforcement learning (learning through rewards).",
    "DL_DEFINITION: Deep learning is a subset of machine learning that utilizes neural networks with multiple layers to model complex patterns in large datasets. It excels in tasks such as image and speech recognition, natural language processing, and game playing, leveraging vast amounts of data and computational power.",
    "ML_APPLICATIONS: Machine learning is widely applied in various fields including finance (fraud detection), healthcare (diagnosis prediction), marketing (customer segmentation), and autonomous vehicles (object detection). Its versatility stems from its ability to adapt to new data and improve over time.",
    "COGNITION_METHODS: Researchers measure human cognition through a combination of standardized psychological tests like the Wechsler Adult Intelligence Scale (WAIS) and Stanford-Binet IQ tests, along with advanced neuroimaging techniques including functional MRI (fMRI) which shows brain activity, electroencephalography (EEG) measuring electrical activity, and positron emission tomography (PET) scans that track metabolic processes.",
    "ROBOT_CAPABILITIES: Contemporary robotics systems can execute both physical operations such as delicate assembly work and microsurgery, as well as cognitive functions including real-time decision-making and environmental interpretation, achieved through sophisticated sensor arrays combined with deep learning algorithms that enable adaptation to dynamic environments.",
    "AI_DEFINITION: Artificial intelligence represents the comprehensive discipline of creating intelligent machines capable of performing tasks that typically require human cognition, including but not limited to machine learning (pattern recognition), natural language processing (communication understanding), computer vision (image interpretation), and robotic control systems (physical interaction)."
])

# Create optimized FAISS index with proper normalization
embeddings = embedder.encode(knowledge_base)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))

In [29]:
def retrieve(question, k, threshold=0.35):
    """Precision retrieval with similarity validation"""
    emb = embedder.encode(question, convert_to_tensor=False)
    emb = np.array([emb]).astype('float32')
    faiss.normalize_L2(emb)
    scores, indices = index.search(emb, k)
    return scores, indices
    # return knowledge_base[indices[0][0]] if scores[0][0] >= threshold else None

In [33]:
question = "What is machine learning?"
scores, indices = retrieve(question, k=4, threshold=0.35)

In [34]:
scores

array([[0.81055415, 0.5807991 , 0.5623052 , 0.5558163 ]], dtype=float32)

In [38]:
indices[0][:3]

array([0, 1, 2], dtype=int64)

In [48]:
context = knowledge_base[indices[0][:3]] if scores[0][0] >= 0.35 else None

In [49]:
context

array(['ML_DEFINITION: Machine learning is a branch of artificial intelligence that develops systems capable of learning from data, identifying patterns, and making decisions with minimal human intervention. Key approaches include supervised learning (using labeled data), unsupervised learning (finding hidden patterns), and reinforcement learning (learning through rewards).',
       'DL_DEFINITION: Deep learning is a subset of machine learning that utilizes neural networks with multiple layers to model complex patterns in large datasets. It excels in tasks such as image and speech recognition, natural language processing, and game playing, leveraging vast amounts of data and computational power.',
       'ML_APPLICATIONS: Machine learning is widely applied in various fields including finance (fraud detection), healthcare (diagnosis prediction), marketing (customer segmentation), and autonomous vehicles (object detection). Its versatility stems from its ability to adapt to new data and 

In [45]:
all(knowledge_base)

True

In [47]:
all(None)

TypeError: 'NoneType' object is not iterable

In [50]:
context[0]

'ML_DEFINITION: Machine learning is a branch of artificial intelligence that develops systems capable of learning from data, identifying patterns, and making decisions with minimal human intervention. Key approaches include supervised learning (using labeled data), unsupervised learning (finding hidden patterns), and reinforcement learning (learning through rewards).'

In [54]:
core_info = context[0].split(":", 1)[1].strip()
if not core_info.endswith("."):
    core_info += "."
    print(core_info[0].upper() + core_info[1:])

In [55]:
core_info

'Machine learning is a branch of artificial intelligence that develops systems capable of learning from data, identifying patterns, and making decisions with minimal human intervention. Key approaches include supervised learning (using labeled data), unsupervised learning (finding hidden patterns), and reinforcement learning (learning through rewards).'